In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

INPUT_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/hakkimizda"
EXTRACTED_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities"
OUTPUT_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

In [3]:
!pip install PyMuPDF
# !pip install groq anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 109.4 MB/s eta 0:00:00


In [ ]:
import json
import fitz  # PyMuPDF
from pathlib import Path
from google.colab import files, userdata
import glob

In [ ]:
# import anthropic
# ANTHROPIC_API_KEY = userdata.get("My_claude_key")
# client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

from groq import Groq
GROQ_API_KEY = userdata.get("My_groq")
client = Groq(api_key=GROQ_API_KEY)

# from openai import OpenAI
# OPENAI_API_KEY = userdata.get("My_OpenAI_API_key")
# client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:

# --- Configuration ---
#MODEL = "claude-sonnet-4-6"
#MODEL = "claude-haiku-4-5"

#Groq_model = "openai/gpt-oss-20b"
groq_model = "openai/gpt-oss-120b"

#gpt_model = "gpt-5.2"

TEMPERATURE = 0

# --- PDF Text Extraction ---
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from a PDF file using PyMuPDF."""
    doc = fitz.open(pdf_path)
    text_parts = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        if text.strip():
            text_parts.append(f"--- Page {page_num + 1} ---\n{text}")
    doc.close()
    full_text = "\n\n".join(text_parts)
    return full_text

def extract_company_name(file_path: str) -> str:
    """Extract company name from filename (e.g., 'hm_about.pdf' -> 'Company H')."""
    name = Path(file_path).stem
    # Clean up common suffixes
    for suffix in ["_about", "_aboutus", "_about_us", "_hakkimizda"]:
        name = name.replace(suffix, "")
    return name.replace("_", " ").replace("-", " ").title()

# --- Build Prompt ---
def build_prompt(document_text):
    """Build the extraction prompt with structured attributes support."""
    prompt = """You are an expert information extraction system. Your task is to extract ALL entities and ALL factual statements from the following company document.

    IMPORTANT RULES:
    - Extract EVERYTHING, not just sustainability-related information. Include business operations, history, products, locations, partnerships, certifications, metrics, goals, people, etc.
    - Stay STRICTLY within what the document states. Do NOT infer, assume, or add any information not explicitly present in the text.
    - For every extracted item, include the exact source sentence from the document so it can be traced back.
    - If something is ambiguous, extract it as stated without interpretation.
    - If the text contains translation artifacts or formatting issues, do your best to interpret the intended meaning but flag any uncertainty.

    Extract the following:

    1. ENTITIES: Every named entity mentioned in the document. Do NOT extract standalone numbers, percentages, dates, or metrics as entities. These belong in factual statements only. Entities must be named things belonging to these categories:
      Categories: Company, Brand/Sub-brand, Person, Location, Certification, Supplier, Organization, Material, Product/Product_Line, Partnership, Technology, Event, Concept/Initiative, Other

      Category notes:
      - Concept/Initiative: Named programs, strategies, campaigns, retail concepts, loyalty programs, sustainability initiatives, projects, or collections
      - Supplier: Named material suppliers, fabric providers, or manufacturing partners
      - Other: Use ONLY when no other category fits. If you find yourself using Other frequently, reconsider whether a more specific category applies.

    2. FACTUAL STATEMENTS: Every discrete factual claim, metric, achievement, commitment, or description.
      Categories:
      - Metric (quantifiable data: percentages, numbers, measurements)
      - Certification (certifications held or pursued)
      - Achievement (completed actions or milestones)
      - Commitment/Goal (future plans, targets, pledges)
      - Process (production methods, supply chain details, operational practices)
      - Partnership (collaborations, memberships, affiliations)
      - History (founding, milestones, timeline events, dates)
      - Product_Info (product descriptions, features, materials)
      - Geographic (locations, markets, operations geography)
      - Organizational (company structure, employee info, governance)
      - Sustainability (environmental or social responsibility claims)
      - Vague_Claim (aspirational or unsubstantiated statements without specific evidence)
      - Other

      IMPORTANT — For each factual statement, extract ALL structured attributes. Attributes are any specific properties or characteristics of an entity. These include but are not limited to:

      Quantitative attributes: counts, percentages, measurements, scores, ratings, years, financial figures
      Material attributes: fabric types, fiber compositions, material sources, material percentages
      Product attributes: color, style, fit, collection name, product category, design features
      Process attributes: production methods, dyeing techniques, washing processes, manufacturing approaches
      Geographic attributes: country of operation, city, number of markets, store locations
      Certification attributes: certification name, certifying body, certification level, year obtained
      Organizational attributes: number of employees, department, role, ownership structure
      Sustainability attributes: emission levels, water usage, energy source, recycling rate, waste type
      Temporal attributes: founding year, launch date, target year, reporting period
      Brand attributes: brand positioning, market segment, target audience, brand values

      For each factual statement provide:
      - subject: The main entity this statement is about
      - attributes: A list of attribute objects extracted from the statement. Each attribute has:
        - attribute_name: Descriptive name (e.g., "fabric_type", "color", "store_count", "certification_name", "production_method", "target_audience", etc.)
        - attribute_value: The specific value
        - attribute_unit: Unit of measurement if applicable, otherwise null
      - If the statement contains NO extractable attributes, set attributes to an empty list [].

        Respond ONLY with valid JSON in this exact format:
        {
        "entities": [
            {
                "name": "entity name",
                "type": "entity category from list above",
                "source_sentence": "exact sentence from document where this entity appears"
            }
        ],
        "factual_statements": [
            {
                "statement": "the discrete factual claim or statement",
                "category": "category from list above",
                "is_verifiable": true or false,
                "subject": "main entity this is about or null",
                "attributes": [
                    {
                        "attribute_name": "descriptive attribute name",
                        "attribute_value": "specific value",
                        "attribute_unit": "unit or null"
                    }
                ],
                "source_sentence": "exact sentence from document containing this statement"
            }
        ]
    }

    EXAMPLES of structured attribute extraction:

    - "Company M has a presence in 34 countries, including Turkiye, the USA, Canada, Germany, and Russia, selling through approximately 4,000 points, including 485 Company M shops." →
      subject: "Company M"
      attributes: [
        {"attribute_name": "country_count", "attribute_value": "34", "attribute_unit": "countries"},
        {"attribute_name": "sales_points", "attribute_value": "4000", "attribute_unit": "points"},
        {"attribute_name": "store_count", "attribute_value": "485", "attribute_unit": "shops"},
        {"attribute_name": "market_countries", "attribute_value": "Turkiye, USA, Canada, Germany, Russia", "attribute_unit": null}
      ]

    - "Hemp Denim, the most sustainable jeans of Company M, was made with hemp fibers that consume minimum water, recycled cotton and bio-based nutshell buttons." →
      subject: "Hemp Denim"
      attributes: [
        {"attribute_name": "fabric_type", "attribute_value": "hemp fibers", "attribute_unit": null},
        {"attribute_name": "secondary_material", "attribute_value": "recycled cotton", "attribute_unit": null},
        {"attribute_name": "button_material", "attribute_value": "bio-based nutshell", "attribute_unit": null},
        {"attribute_name": "water_usage", "attribute_value": "minimum", "attribute_unit": null}
      ]

    - "Dilvin operates in a 5000 m² indoor facility with state-of-the-art systems" →
      subject: "Dilvin"
      attributes: [
        {"attribute_name": "facility_area", "attribute_value": "5000", "attribute_unit": "m²"},
        {"attribute_name": "facility_type", "attribute_value": "indoor", "attribute_unit": null},
        {"attribute_name": "equipment_quality", "attribute_value": "state-of-the-art", "attribute_unit": null}
      ]

    - "Company M is established in the apparel market between the high-end and premium segments" →
      subject: "Company M"
      attributes: [
        {"attribute_name": "market_segment", "attribute_value": "between high-end and premium", "attribute_unit": null}
      ]

    - "Our brand accompanies more women each year with its modern design approach" →
      subject: null, attributes: [] (this is a Vague_Claim with no extractable attributes)

    --- DOCUMENT START ---
    """ + document_text + """
    --- DOCUMENT END ---
    """
    return prompt

def extract_entities_and_facts(client, document_text: str, company_name: str) -> dict:
    """Send document to LLM API and extract entities and factual statements."""
    print(f"\n{'='*60}")
    print(f"Processing: {company_name}")
    print(f"Document length: {len(document_text)} characters")
    print(f"{'='*60}")

    prompt = build_prompt(document_text)

    # # for claude
    # message = client.messages.create(
    #     model=MODEL,
    #     max_tokens=8192,
    #     temperature=TEMPERATURE,
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": prompt
    #         }])
    # response_text = message.content[0].text

    # # for GPT
    # message = client.chat.completions.create(
    #     model=groq_model,
    #     max_completion_tokens=32768, # for gpt_5.2
    #     temperature=TEMPERATURE,
    #     messages=[
    #         {"role": "user", "content": prompt}
    # ])
    # response_text = message.choices[0].message.content

    import time

    message = client.chat.completions.create(
        model=groq_model,
        max_tokens=62768,
        temperature=TEMPERATURE,
        timeout=120,  # 120 seconds instead of default
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    response_text = message.choices[0].message.content

    try:
        extracted_data = json.loads(response_text)
    except json.JSONDecodeError:
        # Try fixing truncated JSON by closing open brackets
        open_braces = response_text.count('{') - response_text.count('}')
        open_brackets = response_text.count('[') - response_text.count(']')
        response_text = response_text.rstrip().rstrip(',')
        response_text += ']' * open_brackets
        response_text += '}' * open_braces

        try:
            extracted_data = json.loads(response_text)
            print(f"  JSON repaired successfully.")
        except json.JSONDecodeError as e:
            print(f"  WARNING: JSON repair failed for {company_name}: {e}")
            extracted_data = {
                "entities": [],
                "factual_statements": [],
                "raw_response": response_text,
                "parse_error": str(e)
            }

    # Add metadata
    extracted_data["company_name"] = company_name
    extracted_data["entity_count"] = len(extracted_data.get("entities", []))
    extracted_data["statement_count"] = len(extracted_data.get("factual_statements", []))

    print(f"  ✅ Entities extracted: {extracted_data['entity_count']}")
    print(f"  ✅ Factual statements extracted: {extracted_data['statement_count']}")

    # Print category breakdown
    if extracted_data.get("factual_statements"):
        categories = {}
        for stmt in extracted_data["factual_statements"]:
            cat = stmt.get("category", "Unknown")
            categories[cat] = categories.get(cat, 0) + 1
        print(f"  📊 Statement categories:")
        for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
            print(f"      {cat}: {count}")

    return extracted_data

def save_output(data: dict, output_path: str):
    """Save extracted data as formatted JSON."""
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  💾 Saved to: {output_path}")

In [ ]:
# --- Main Processing ---
import glob

pdf_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdf")))

if not pdf_files:
    print(f"❌ ERROR: No PDF files found in '{INPUT_DIR}'")
else:
    print(f"Found {len(pdf_files)} PDF file(s) to process:")
    for f in pdf_files:
        print(f"  - {os.path.basename(f)}")

    all_results = {}

    for file_path in pdf_files:
        company_name = extract_company_name(file_path)

        # Step A: Extract text from PDF
        print(f"\n📄 Extracting text from {os.path.basename(file_path)}...")
        document_text = extract_text_from_pdf(file_path)

        if not document_text.strip():
            print(f"  ⚠️ WARNING: No text extracted from {file_path}. ")
            continue

        print(f"  Extracted {len(document_text)} characters of text.")

        # Step B: Send to Claude for entity/fact extraction
        extracted = extract_entities_and_facts(client, document_text, company_name)

        # Step C: Save individual company output
        output_filename = f"{Path(file_path).stem}_extracted.json"
        output_path = os.path.join(EXTRACTED_DIR, output_filename)
        save_output(extracted, output_path)

        all_results[company_name] = extracted

    # Save combined output
    combined_path = os.path.join(EXTRACTED_DIR, "all_companies_combined.json")
    save_output(all_results, combined_path)

    # Print summary
    print(f"\n{'='*60}")
    print("📋 EXTRACTION COMPLETE - SUMMARY")
    print(f"{'='*60}")
    total_entities = 0
    total_statements = 0
    for company, data in all_results.items():
        e_count = data.get("entity_count", 0)
        s_count = data.get("statement_count", 0)
        total_entities += e_count
        total_statements += s_count
        print(f"  {company}: {e_count} entities, {s_count} statements")
    print(f"  TOTAL: {total_entities} entities, {total_statements} statements")
    print(f"\nOutputs saved to: {EXTRACTED_DIR}/")

    print(f"\n{'='*60}")
    print("⚠️  IMPORTANT: HUMAN VERIFICATION NEEDED")
    print(f"{'='*60}")
    print("Before proceeding to Step 3 (Knowledge Graph construction),")
    print("please spot-check the extracted JSON files to verify:")
    print("  1. No entities or facts were hallucinated (not in source)")
    print("  2. No important information was missed")
    print("  3. Categories are assigned correctly")
    print("  4. Source sentences match the original documents")

Found 4 PDF file(s) to process:
  - company_f.pdf
  - company_g.pdf
  - company_h.pdf
  - company_m.pdf

📄 Extracting text from company_f.pdf...
  Extracted 1957 characters of text.

Processing: Company F En
Document length: 1957 characters
  ✅ Entities extracted: 11
  ✅ Factual statements extracted: 15
  📊 Statement categories:
      Process: 5
      Vague_Claim: 4
      History: 2
      Metric: 2
      Commitment: 1
      Product_Info: 1
  💾 Saved to: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/company_f_extracted.json

📄 Extracting text from company_g.pdf...
  Extracted 1405 characters of text.

Processing: Company G En
Document length: 1405 characters
  ✅ Entities extracted: 13
  ✅ Factual statements extracted: 12
  📊 Statement categories:
      Vague_Claim: 5
      History: 2
      Process: 2
      Commitment/Goal: 1
      Organizational: 1
      Achievement: 1
  💾 Saved to: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/compa

In [ ]:
# # Download individual files
# import glob
# for json_file in glob.glob(os.path.join(EXTRACTED_DIR, "*.json")):
#     files.download(json_file)

# print("✅ All JSON files downloaded.")

In [ ]:
"""
Function to Export Extracted Entities, Factual Statements with attributes to CSV
"""
import os
import json
import glob
from pathlib import Path
import csv

def export_entities(company_data, output_dir):
    """Export entities to CSV — one per company plus combined."""
    os.makedirs(output_dir, exist_ok=True)

    all_rows = []

    for company_name, data in company_data.items():
        safe_name = company_name.lower().replace(" ", "_")
        entities = data.get("entities", [])

        rows = []
        for ent in entities:
            row = {
                "company": company_name,
                "name": ent.get("name", ""),
                "type": ent.get("type", ""),
                "source_sentence": ent.get("source_sentence", "")
            }
            rows.append(row)
            all_rows.append(row)

        # Per-company CSV
        if rows:
            path = os.path.join(output_dir, f"{safe_name}_entities.csv")
            fieldnames = list(rows[0].keys())
            with open(path, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
            print(f"Saved: {path} ({len(rows)} entities)")

    # Combined CSV
    if all_rows:
        path = os.path.join(output_dir, "all_entities.csv")
        fieldnames = list(all_rows[0].keys())
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\nCombined: {path} ({len(all_rows)} total entities)")

def export_statements_with_attributes(company_data, output_dir):
    """Export factual statements with structured attributes (list format) to CSV."""
    import os
    import csv
    os.makedirs(output_dir, exist_ok=True)

    all_rows = []

    for company_name, data in company_data.items():
        safe_name = company_name.lower().replace(" ", "_")
        statements = data.get("factual_statements", [])

        rows = []
        for stmt in statements:
            attributes = stmt.get("attributes", [])

            if attributes:
                # One row per attribute
                for attr in attributes:
                    row = {
                        "company": company_name,
                        "statement": stmt.get("statement", ""),
                        "category": stmt.get("category", ""),
                        "is_verifiable": stmt.get("is_verifiable", ""),
                        "subject": stmt.get("subject", ""),
                        "attribute_name": attr.get("attribute_name", ""),
                        "attribute_value": attr.get("attribute_value", ""),
                        "attribute_unit": attr.get("attribute_unit", ""),
                        "source_sentence": stmt.get("source_sentence", "")
                    }
                    rows.append(row)
                    all_rows.append(row)
            else:
                # No attributes — still include the statement with empty attribute fields
                row = {
                    "company": company_name,
                    "statement": stmt.get("statement", ""),
                    "category": stmt.get("category", ""),
                    "is_verifiable": stmt.get("is_verifiable", ""),
                    "subject": stmt.get("subject", ""),
                    "attribute_name": "",
                    "attribute_value": "",
                    "attribute_unit": "",
                    "source_sentence": stmt.get("source_sentence", "")
                }
                rows.append(row)
                all_rows.append(row)

        # Per-company CSV
        if rows:
            path = os.path.join(output_dir, f"{safe_name}_statements_with_attrs.csv")
            fieldnames = list(rows[0].keys())
            with open(path, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(rows)
            print(f"Saved: {path} ({len(rows)} rows)")

    # Combined CSV
    if all_rows:
        path = os.path.join(output_dir, "all_statements_with_attrs.csv")
        fieldnames = list(all_rows[0].keys())
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\nCombined: {path} ({len(all_rows)} total rows)")

In [ ]:
# run function

CSV_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv"
os.makedirs(CSV_DIR, exist_ok=True)

# --- Load all company JSON files ---
company_data = {}
json_files = sorted(glob.glob(os.path.join(EXTRACTED_DIR, "*_extracted.json")))

for filepath in json_files:
    # Skip the combined file if it exists
    if "all_companies" in filepath:
        continue
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    company_name = data.get("company_name", Path(filepath).stem)
    company_data[company_name] = data
    print(f"Loaded: {company_name}")

print(f"\nTotal companies loaded: {len(company_data)}")

# --- Export ---
export_entities(company_data, CSV_DIR)
export_statements_with_attributes(company_data, CSV_DIR)

Loaded: Company F En
Loaded: Company G En
Loaded: Company H
Loaded: Company M En

Total companies loaded: 4
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/company_f_entities.csv (11 entities)
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/company_g_entities.csv (13 entities)
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/hm_entities.csv (72 entities)
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/company_m_entities.csv (101 entities)

Combined: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/all_entities.csv (197 total entities)
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/company_f_statements_with_attrs.csv (36 rows)
Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv/comp

In [ ]:
# For KG construction, Load all individual company JSON files (skip the combined file)
company_data = {}
json_files = sorted(glob.glob(os.path.join(EXTRACTED_DIR, "*_extracted.json")))

for filepath in json_files:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    company_name = data.get("company_name", os.path.basename(filepath))
    company_data[company_name] = data
    e = data.get("entity_count", len(data.get("entities", [])))
    s = data.get("statement_count", len(data.get("factual_statements", [])))
    print(f"Loaded: {company_name} — {e} entities, {s} statements")

print(f"\nTotal companies loaded: {len(company_data)}")


Loaded: Company F En — 11 entities, 15 statements
Loaded: Company G En — 13 entities, 12 statements
Loaded: Company H — 59 entities, 60 statements
Loaded: Company M En — 101 entities, 76 statements

Total companies loaded: 4


In [ ]:
# ============================================================
# CELL 3: Build separate knowledge graphs
# ============================================================

import networkx as nx
import re

def clean_name(name):
    """Normalize entity names for consistent matching."""
    name = name.strip()
    name = re.sub(r'\s+', ' ', name)
    return name

def find_related_entities(statement_text, entities_list):
    """
    Find which entities are mentioned in a factual statement.
    Returns a list of entity names found in the statement.
    """
    mentioned = []
    statement_lower = statement_text.lower()
    for entity in entities_list:
        entity_name = entity["name"]
        if len(entity_name) <= 3:
            continue
        if entity_name.lower() in statement_lower:
            mentioned.append(clean_name(entity_name))
    return mentioned

def build_company_graph(company_name, data):
    """
    Build a knowledge graph for a single company.
    Now includes structured attributes on nodes and edges.
    """
    G = nx.DiGraph()
    entities = data.get("entities", [])
    statements = data.get("factual_statements", [])

    # Determine the main company node name
    main_company = company_name
    for ent in entities:
        if ent["type"] == "Company":
            main_company = clean_name(ent["name"])
            break
    if main_company == company_name:
        for ent in entities:
            if ent["type"] == "Brand/Sub-brand":
                main_company = clean_name(ent["name"])
                break

    G.graph["main_company"] = main_company
    G.graph["company_key"] = company_name

    # --- Add entity nodes ---
    for entity in entities:
        node_name = clean_name(entity["name"])
        if G.has_node(node_name):
            existing_type = G.nodes[node_name].get("type", "")
            if entity["type"] not in existing_type:
                G.nodes[node_name]["type"] = existing_type + ", " + entity["type"] if existing_type else entity["type"]
        else:
            G.add_node(
                node_name,
                type=entity["type"],
                source_sentence=entity["source_sentence"],
                attributes={}  # Will store structured attributes
            )

    # --- Add factual statement edges with structured attributes ---
    for idx, stmt in enumerate(statements):
        statement_text = stmt["statement"]
        category = stmt["category"]
        is_verifiable = stmt.get("is_verifiable", False)
        source_sentence = stmt["source_sentence"]

        # Structured attribute fields
        subject = stmt.get("subject", None)
        attr_name = stmt.get("attribute_name", None)
        attr_value = stmt.get("attribute_value", None)
        attr_unit = stmt.get("attribute_unit", None)

        # Build structured attribute dict if available
        structured_attr = None
        if attr_name and attr_value:
            structured_attr = {
                "attribute_name": attr_name,
                "attribute_value": attr_value,
                "attribute_unit": attr_unit,
                "source_statement": statement_text,
                "is_verifiable": is_verifiable
            }

        # If we have a subject and structured attribute, attach it to the subject node
        if subject and structured_attr:
            subject_clean = clean_name(subject)
            # Find the matching node (might be slightly different name)
            matched_node = None
            for node in G.nodes():
                if node.lower() == subject_clean.lower():
                    matched_node = node
                    break
                if subject_clean.lower() in node.lower() or node.lower() in subject_clean.lower():
                    matched_node = node
                    break

            if matched_node:
                # Attach structured attribute to the node
                if "attributes" not in G.nodes[matched_node]:
                    G.nodes[matched_node]["attributes"] = {}
                G.nodes[matched_node]["attributes"][attr_name] = {
                    "value": attr_value,
                    "unit": attr_unit,
                    "verifiable": is_verifiable,
                    "source": statement_text
                }

        # --- Edge creation (same logic as before, with structured attrs added) ---
        mentioned_entities = find_related_entities(statement_text, entities)

        # Common edge attributes
        edge_attrs = {
            "relationship": "has_claim",
            "category": category,
            "is_verifiable": is_verifiable,
            "statement": statement_text
        }
        if structured_attr:
            edge_attrs["structured_attribute"] = structured_attr

        if len(mentioned_entities) == 0:
            stmt_node_id = f"STMT_{idx}"
            node_attrs = {
                "type": "FactualStatement",
                "statement": statement_text,
                "category": category,
                "is_verifiable": is_verifiable,
                "source_sentence": source_sentence
            }
            if structured_attr:
                node_attrs["structured_attribute"] = structured_attr
            G.add_node(stmt_node_id, **node_attrs)
            G.add_edge(main_company, stmt_node_id, **edge_attrs)

        elif len(mentioned_entities) == 1:
            target = mentioned_entities[0]
            if target == main_company:
                stmt_node_id = f"STMT_{idx}"
                node_attrs = {
                    "type": "FactualStatement",
                    "statement": statement_text,
                    "category": category,
                    "is_verifiable": is_verifiable,
                    "source_sentence": source_sentence
                }
                if structured_attr:
                    node_attrs["structured_attribute"] = structured_attr
                G.add_node(stmt_node_id, **node_attrs)
                G.add_edge(main_company, stmt_node_id, **edge_attrs)
            else:
                edge_attrs["relationship"] = "related_to"
                G.add_edge(main_company, target, **edge_attrs)

        else:
            for i in range(len(mentioned_entities)):
                for j in range(i + 1, len(mentioned_entities)):
                    ea = edge_attrs.copy()
                    ea["relationship"] = "related_to"
                    G.add_edge(mentioned_entities[i], mentioned_entities[j], **ea)
            for ent_name in mentioned_entities:
                if ent_name != main_company and not G.has_edge(main_company, ent_name):
                    ea = edge_attrs.copy()
                    ea["relationship"] = "mentions"
                    G.add_edge(main_company, ent_name, **ea)

    return G


# --- Build all graphs ---
company_graphs = {}

for company_name, data in company_data.items():
    G = build_company_graph(company_name, data)
    company_graphs[company_name] = G

    main = G.graph["main_company"]
    print(f"\n{'='*50}")
    print(f"Company: {company_name} (main node: {main})")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")

    # Node type breakdown
    ntypes = {}
    for node, attrs in G.nodes(data=True):
        t = attrs.get("type", "Unknown")
        ntypes[t] = ntypes.get(t, 0) + 1
    for t, c in sorted(ntypes.items(), key=lambda x: -x[1]):
        print(f"    {t}: {c}")

    # Verifiable vs non-verifiable statements
    verifiable = sum(1 for _, _, a in G.edges(data=True) if a.get("is_verifiable", False))
    non_verifiable = G.number_of_edges() - verifiable
    print(f"  Verifiable edges: {verifiable}")
    print(f"  Non-verifiable edges: {non_verifiable}")

# Overall summary
print(f"\n{'='*50}")
print(f"ALL GRAPHS BUILT SUCCESSFULLY")
print(f"{'='*50}")
total_nodes = sum(G.number_of_nodes() for G in company_graphs.values())
total_edges = sum(G.number_of_edges() for G in company_graphs.values())
print(f"Total across all companies: {total_nodes} nodes, {total_edges} edges")


Company: Company F En (main node: Company F)
  Nodes: 15
  Edges: 19
    FactualStatement: 5
    Concept/Initiative: 4
    Product/Product_Line: 3
    Person, Company: 1
    Location: 1
    Target_Group: 1
  Verifiable edges: 15
  Non-verifiable edges: 4

Company: Company G En (main node: Company G)
  Nodes: 20
  Edges: 55
    Location: 11
    FactualStatement: 7
    Company: 1
    Concept/Initiative: 1
  Verifiable edges: 2
  Non-verifiable edges: 53

Company: Company H (main node: Company H Group)
  Nodes: 76
  Edges: 115
    Location: 17
    FactualStatement: 17
    Brand/Sub-brand: 14
    Concept/Initiative: 7
    Person: 6
    Organization: 5
    Company: 4
    Certification: 2
    Supplier: 1
    Event: 1
    Product/Product_Line: 1
    Partnership: 1
  Verifiable edges: 114
  Non-verifiable edges: 1

Company: Company M En (main node: Company M)
  Nodes: 116
  Edges: 154
    Organization: 24
    Product/Product_Line: 19
    FactualStatement: 15
    Person: 14
    Location: 13
  

In [ ]:
# ============================================================
# CELL 5: Export all graphs for later use in KG-RAG
# ============================================================

import pickle

for company_name, G in company_graphs.items():
    safe_name = company_name.lower().replace(" ", "_")

    # --- Pickle export ---
    pickle_path = os.path.join(OUTPUT_DIR, f"kg_{safe_name}.gpickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(G, f)

    # --- JSON export ---
    graph_json = {
        "company_name": company_name,
        "main_company": G.graph.get("main_company", company_name),
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "nodes": [],
        "edges": []
    }
    for node, attrs in G.nodes(data=True):
        node_data = {"id": node}
        node_data.update(attrs)
        graph_json["nodes"].append(node_data)
    for u, v, attrs in G.edges(data=True):
        edge_data = {"source": u, "target": v}
        edge_data.update(attrs)
        graph_json["edges"].append(edge_data)

    json_path = os.path.join(OUTPUT_DIR, f"kg_{safe_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(graph_json, f, indent=2, ensure_ascii=False)

    print(f"Exported {company_name}: {pickle_path} + {json_path}")

# --- Combined stats file ---
all_stats = {}
for company_name, G in company_graphs.items():
    ntypes = {}
    for node, attrs in G.nodes(data=True):
        t = attrs.get("type", "Unknown")
        ntypes[t] = ntypes.get(t, 0) + 1
    ecats = {}
    for u, v, attrs in G.edges(data=True):
        c = attrs.get("category", "Unknown")
        ecats[c] = ecats.get(c, 0) + 1

    all_stats[company_name] = {
        "main_company": G.graph.get("main_company", company_name),
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "node_types": ntypes,
        "edge_categories": ecats,
        "verifiable_edges": sum(1 for _, _, a in G.edges(data=True) if a.get("is_verifiable", False)),
        "non_verifiable_edges": sum(1 for _, _, a in G.edges(data=True) if not a.get("is_verifiable", False))
    }

stats_path = os.path.join(OUTPUT_DIR, "all_graphs_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(all_stats, f, indent=2, ensure_ascii=False)

print(f"\nAll stats saved: {stats_path}")
print(f"\n{'='*50}")
print("STEP 3 COMPLETE — SEPARATE GRAPHS PER COMPANY")
print(f"{'='*50}")
for company_name, G in company_graphs.items():
    print(f"  {G.graph['main_company']}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nAll files saved to: {OUTPUT_DIR}")
print(f"\nNext step: Use each company's graph as the retrieval source")
print(f"for KG-RAG to generate and test sustainability marketing content.")

Exported Company F En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_f.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_f.json
Exported Company G En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_g.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_g.json
Exported Company H: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_hm.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_hm.json
Exported Company M En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_m.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_company_m.json

All stats saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/all_graphs_stats.json

STEP 3 COMPLETE — SEPARATE GRAPHS PER COMPANY
  Company F: 15 nodes, 19 edges
  C

In [ ]:
!pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Interactive PyVis Visualizations Per Company KG
# ============================================================

from pyvis.network import Network
import os
import pickle
import glob
from pathlib import Path

# --- Paths ---
KG_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs"
VIS_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations"
os.makedirs(VIS_DIR, exist_ok=True)

# --- Color mapping ---
COLOR_MAP = {
    "Company": "#FF6B6B",
    "Brand/Sub-brand": "#FF8E8E",
    "Person": "#4ECDC4",
    "Location": "#45B7D1",
    "Certification": "#96CEB4",
    "Organization": "#FFEAA7",
    "Material": "#DDA0DD",
    "Product/Product_Line": "#98D8C8",
    "Partnership": "#F7DC6F",
    "Technology": "#BB8FCE",
    "Event": "#85C1E9",
    "Supplier": "#F1948A",
    "Concept/Initiative": "#73C6B6",
    "FactualStatement": "#D5D8DC",
    "Other": "#AEB6BF",
}

def get_node_color(node_type):
    for key, color in COLOR_MAP.items():
        if key in str(node_type):
            return color
    return "#AEB6BF"

# --- Load and visualize each company KG ---
for pkl_file in sorted(glob.glob(os.path.join(KG_DIR, "*.gpickle"))):
    with open(pkl_file, "rb") as f:
        G = pickle.load(f)

    company_key = G.graph.get("company_key", Path(pkl_file).stem)
    main_company = G.graph.get("main_company", company_key)

    print(f"\nBuilding visualization for: {main_company} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

    net = Network(
        height="800px",
        width="100%",
        bgcolor="#ffffff",
        font_color="#333333",
        notebook=True,
        cdn_resources="remote"
    )

    # --- Add nodes ---
    for node, attrs in G.nodes(data=True):
        node_type = attrs.get("type", "Other")
        color = get_node_color(node_type)

        if node.startswith("STMT_"):
            # Statement nodes — show shortened text
            statement = attrs.get("statement", node)
            label = statement[:40] + "..." if len(statement) > 40 else statement

            # Build tooltip with full details
            title_parts = [
                f"Type: FactualStatement",
                f"Category: {attrs.get('category', 'N/A')}",
                f"Verifiable: {attrs.get('is_verifiable', 'N/A')}",
                f"",
                f"Full statement: {statement}"
            ]
            # Add structured attributes if present
            sa = attrs.get("structured_attribute", None)
            if sa:
                title_parts.append(f"")
                title_parts.append(f"Attribute: {sa.get('attribute_name', '')}")
                title_parts.append(f"Value: {sa.get('attribute_value', '')} {sa.get('attribute_unit', '') or ''}")

            title = "\n".join(title_parts)
            size = 12

        else:
            # Entity nodes
            label = node
            title_parts = [
                f"Type: {node_type}",
                f"Source: {attrs.get('source_sentence', 'N/A')}"
            ]
            # Show structured attributes stored on node
            node_attrs = attrs.get("attributes", {})
            if node_attrs:
                title_parts.append("")
                title_parts.append("Structured Attributes:")
                for attr_name, attr_data in node_attrs.items():
                    val = attr_data.get("value", "?")
                    unit = attr_data.get("unit", "")
                    unit_str = f" {unit}" if unit else ""
                    ver = "✓" if attr_data.get("verifiable", False) else "?"
                    title_parts.append(f"  {attr_name}: {val}{unit_str} [{ver}]")

            title = "\n".join(title_parts)

            # Larger size for main company node
            if node_type in ["Company", "Brand/Sub-brand"]:
                size = 35
            elif node_type in ["Certification", "Concept/Initiative"]:
                size = 25
            else:
                size = 18

        net.add_node(node, label=label, title=title, color=color, size=size)

    # --- Add edges ---
    for u, v, attrs in G.edges(data=True):
        relationship = attrs.get("relationship", "related_to")
        category = attrs.get("category", "")
        statement = attrs.get("statement", "")
        is_verifiable = attrs.get("is_verifiable", False)
        ver_tag = "✓ Verified" if is_verifiable else "? Unverified"

        title_parts = [
            f"Relationship: {relationship}",
            f"Category: {category}",
            f"Status: {ver_tag}",
            f"",
            f"Statement: {statement}"
        ]
        sa = attrs.get("structured_attribute", None)
        if sa:
            title_parts.append(f"")
            title_parts.append(f"Attribute: {sa.get('attribute_name', '')} = {sa.get('attribute_value', '')} {sa.get('attribute_unit', '') or ''}")

        title = "\n".join(title_parts)

        # Color edges by verifiability
        edge_color = "#2ECC71" if is_verifiable else "#E74C3C"

        net.add_edge(u, v, title=title, color=edge_color, width=1.5)

    # --- Physics settings for better layout ---
    net.set_options("""
    {
        "physics": {
            "forceAtlas2Based": {
                "gravitationalConstant": -100,
                "centralGravity": 0.01,
                "springLength": 200,
                "springConstant": 0.02
            },
            "solver": "forceAtlas2Based",
            "stabilization": {
                "iterations": 200
            }
        },
        "interaction": {
            "hover": true,
            "tooltipDelay": 100,
            "navigationButtons": true
        }
    }
    """)

    # --- Save ---
    html_path = os.path.join(VIS_DIR, f"kg_{company_key.lower().replace(' ', '_')}_interactive.html")
    net.show(html_path)
    print(f"  Saved: {html_path}")

print(f"\n{'='*50}")
print(f"All interactive visualizations saved to: {VIS_DIR}")
print(f"Open the HTML files in your browser to explore.")
print(f"  Green edges = verified facts")
print(f"  Red edges = unverified claims")
print(f"  Hover over nodes/edges to see details and attributes")


Building visualization for: Company F (15 nodes, 19 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_company_f_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_company_f_interactive.html

Building visualization for: Company G (20 nodes, 55 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_company_g_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_company_g_interactive.html

Building visualization for: Company H Group (76 nodes, 115 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_hm_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_hm_interactive.html

Building visualization for: Company M (116 nodes, 154 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_company_m_interactive.html
  Saved: /content/drive/